In [251]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt

# Content-Based Recommender System
1. Represent each discount as an item-feature vector.
2. Represent each user as a weighted combination of discount vectors.
3. Calculate cosine similarity between user profiles and discount profiles.
4. Generate top-N recommendations for each user.

In [252]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [253]:
path = Path("../data/raw/")

users_df = pd.read_csv(path / "users.csv")
categories_df = pd.read_csv(path / "categories_.csv")
category_discount_df = pd.read_csv( path / "category_discount_.csv")
countries_df = pd.read_csv(path / "countries_.csv")
discount_button_click_statistics_df = pd.read_csv(path / "discount_button_click_statistics_.csv")
discount_favourites_df = pd.read_csv(path / "discount_favourites_.csv")
discount_rates_df = pd.read_csv(path / "discount_rates_.csv")
discounts_df = pd.read_csv(path / "discounts.csv")
localization_default_df = pd.read_csv(path / "localization_default_.csv")
location_discount_df = pd.read_csv(path / "location_discount_.csv")
locations_df = pd.read_csv(path / "locations_.csv")
roles_df = pd.read_csv(path / "roles_.csv")
users_assigned_locations_df = pd.read_csv(path / "users_assigned_locations_.csv")


interaction_clean = pd.read_csv(path /"interaction_clean.csv")


In [254]:
interaction_clean.head()

,user_id,discount_id,event_type,event_time,interaction_weight,source_table,location_id,role_id
0,1535,17,SHOW_AT_LOCATION,2025-01-16 15:18:53.386,0.1,discount_button_click_statistics,16.0,3.0
1,9,522,SHOW_AT_LOCATION,2025-01-16 15:39:03.441,0.1,discount_button_click_statistics,16.0,3.0
2,1455,5,SHOW_AT_LOCATION,2025-01-16 17:59:54.167,0.1,discount_button_click_statistics,16.0,3.0
3,1455,2,SHOW_AT_LOCATION,2025-01-16 18:00:05.394,0.1,discount_button_click_statistics,16.0,3.0
4,1418,11,SHOW_AT_LOCATION,2025-01-16 19:02:26.925,0.1,discount_button_click_statistics,16.0,3.0


In [255]:
interaction_clean.head()

,user_id,discount_id,event_type,event_time,interaction_weight,source_table,location_id,role_id
0,1535,17,SHOW_AT_LOCATION,2025-01-16 15:18:53.386,0.1,discount_button_click_statistics,16.0,3.0
1,9,522,SHOW_AT_LOCATION,2025-01-16 15:39:03.441,0.1,discount_button_click_statistics,16.0,3.0
2,1455,5,SHOW_AT_LOCATION,2025-01-16 17:59:54.167,0.1,discount_button_click_statistics,16.0,3.0
3,1455,2,SHOW_AT_LOCATION,2025-01-16 18:00:05.394,0.1,discount_button_click_statistics,16.0,3.0
4,1418,11,SHOW_AT_LOCATION,2025-01-16 19:02:26.925,0.1,discount_button_click_statistics,16.0,3.0


In [256]:
interaction_clean.shape

(34606, 8)

In [257]:
interaction_clean.isna().sum().sort_values(ascending=False)

location_id           3198
role_id               3198
user_id                  0
discount_id              0
event_type               0
event_time               0
interaction_weight       0
source_table             0
dtype: int64

In [258]:
interaction_clean['event_type'].value_counts()

event_type
OPEN_CARD           19253
GET_A_PROMOCODE      9930
SHOW_AT_LOCATION     2225
FAVOURITE            1652
LIKE                 1515
DISLIKE                31
Name: count, dtype: int64

In [259]:
interaction_clean['interaction_weight'].describe()

count    34606.000000
mean         1.066821
std          1.107379
min         -3.000000
25%          0.300000
50%          0.300000
75%          2.000000
max          4.000000
Name: interaction_weight, dtype: float64

### Step 3: Build Item Raw Feature Table

In [260]:
discounts_df.columns

Index(['id', 'company_id', 'type', 'discount_condition', 'discount_type',
       'start_date', 'end_date', 'image', 'size_min', 'size_max', 'like_count',
       'dislike_count', 'short_description', 'time_create', 'creator_user_id',
       'discount_name', 'is_archived', 'fixed_size'],
      dtype='object')

In [261]:
discount_base = discounts_df[[
    'id',
    'type',
    'discount_type',
    'size_min',
    'size_max',
    'like_count',
    'dislike_count',
    'is_archived'
]].copy()


In [262]:
discount_base = discount_base.rename(columns={
    'id':'discount_id'
})

In [263]:
discount_base.head()

,discount_id,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived
0,785,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False
1,613,нутріциолог,PROMOCODE,15.0,15.0,2,0,False
2,765,English School,PROMOCODE,20.0,20.0,1,0,False
3,47,Sightseeing,PROMOCODE,10.0,10.0,0,0,False
4,215,Cafe,PROMOCODE,5.0,5.0,0,0,False


In [264]:
discounts_df['is_archived'].value_counts(dropna=False)

is_archived
False    909
True     107
Name: count, dtype: int64

In [265]:
discount_base = discount_base[
    discount_base["is_archived"] == False
].copy()

In [266]:
discount_base['is_archived'].value_counts()

is_archived
False    909
Name: count, dtype: int64

In [267]:
discounts_df['is_archived'].value_counts(dropna=False)

is_archived
False    909
True     107
Name: count, dtype: int64

In [268]:
discounts_df.head()

,id,company_id,type,discount_condition,discount_type,start_date,end_date,image,size_min,size_max,like_count,dislike_count,short_description,time_create,creator_user_id,discount_name,is_archived,fixed_size
0,785,1144,shoe and accessory cleaning,Special Offer for Andersen Employees: 10% Off ...,PROMOCODE,2025-06-09,NaN,https://andersen-benefits.s3.amazonaws.com/rem...,10.0,10.0,1,0,SHINEBABY10,2025-06-09 14:07:16.800,15,NaN,False,NaN
1,613,459,нутріциолог,Нутриціолог із медичною освітою та понад восьм...,PROMOCODE,2025-02-21,NaN,https://andersen-benefits.s3.amazonaws.com/Spe...,15.0,15.0,2,0,Andersen. Для отримання напишіть в Telegram: @...,2025-02-21 11:13:11.939,46,NaN,False,NaN
2,765,1116,English School,20% discount on all professional English cours...,PROMOCODE,2025-05-26,NaN,https://andersen-benefits.s3.amazonaws.com/192...,20.0,20.0,1,0,ANDERSEN20,2025-05-26 11:46:19.027,46,NaN,False,NaN
3,47,48,Sightseeing,Pass providing,PROMOCODE,2024-03-01,NaN,https://andersen-benefits.s3.amazonaws.com/par...,10.0,10.0,0,0,Pass providing,NaN,15,NaN,False,NaN
4,215,82,Cafe,By promo code,PROMOCODE,2025-05-20,NaN,https://andersen-benefits.s3.amazonaws.com/905...,5.0,5.0,0,0,promocode - Andersen5,NaN,15,NaN,False,NaN


In [269]:
discounts_df.columns

Index(['id', 'company_id', 'type', 'discount_condition', 'discount_type',
       'start_date', 'end_date', 'image', 'size_min', 'size_max', 'like_count',
       'dislike_count', 'short_description', 'time_create', 'creator_user_id',
       'discount_name', 'is_archived', 'fixed_size'],
      dtype='object')

## 4. Create Category Features


In [270]:
category_discount_df.head()

,id,category_id,discount_id
0,1,1,1
1,2,1,2
2,3,3,3
3,4,2,4
4,5,1,5


In [271]:
discounts_df.head()

,id,company_id,type,discount_condition,discount_type,start_date,end_date,image,size_min,size_max,like_count,dislike_count,short_description,time_create,creator_user_id,discount_name,is_archived,fixed_size
0,785,1144,shoe and accessory cleaning,Special Offer for Andersen Employees: 10% Off ...,PROMOCODE,2025-06-09,NaN,https://andersen-benefits.s3.amazonaws.com/rem...,10.0,10.0,1,0,SHINEBABY10,2025-06-09 14:07:16.800,15,NaN,False,NaN
1,613,459,нутріциолог,Нутриціолог із медичною освітою та понад восьм...,PROMOCODE,2025-02-21,NaN,https://andersen-benefits.s3.amazonaws.com/Spe...,15.0,15.0,2,0,Andersen. Для отримання напишіть в Telegram: @...,2025-02-21 11:13:11.939,46,NaN,False,NaN
2,765,1116,English School,20% discount on all professional English cours...,PROMOCODE,2025-05-26,NaN,https://andersen-benefits.s3.amazonaws.com/192...,20.0,20.0,1,0,ANDERSEN20,2025-05-26 11:46:19.027,46,NaN,False,NaN
3,47,48,Sightseeing,Pass providing,PROMOCODE,2024-03-01,NaN,https://andersen-benefits.s3.amazonaws.com/par...,10.0,10.0,0,0,Pass providing,NaN,15,NaN,False,NaN
4,215,82,Cafe,By promo code,PROMOCODE,2025-05-20,NaN,https://andersen-benefits.s3.amazonaws.com/905...,5.0,5.0,0,0,promocode - Andersen5,NaN,15,NaN,False,NaN


In [272]:
discount_categories = category_discount_df[[
    'discount_id',
    'category_id'
]]


In [273]:
discount_categories.head(3)

,discount_id,category_id
0,1,1
1,2,1
2,3,3


In [274]:
discount_categories.isna().sum()

discount_id    0
category_id    0
dtype: int64

In [275]:
category_features = pd.get_dummies(discount_categories['category_id'],prefix='category')

In [276]:
category_features.head()

,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12
0,True,False,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,False,False
2,False,False,True,False,False,False,False,False,False,False,False,False
3,False,True,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,False,False


In [277]:
discount_category_matrix = pd.concat(
    [
        discount_categories[['discount_id']],category_features
    ],
    axis=1
)

In [278]:
discount_categories.head()

,discount_id,category_id
0,1,1
1,2,1
2,3,3
3,4,2
4,5,1


In [279]:
discount_category_matrix.head()

,discount_id,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12
0,1,True,False,False,False,False,False,False,False,False,False,False,False
1,2,True,False,False,False,False,False,False,False,False,False,False,False
2,3,False,False,True,False,False,False,False,False,False,False,False,False
3,4,False,True,False,False,False,False,False,False,False,False,False,False
4,5,True,False,False,False,False,False,False,False,False,False,False,False


In [280]:
discount_category_matrix = (discount_category_matrix.groupby('discount_id').max().reset_index())

In [281]:
discount_category_matrix.head()

,discount_id,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12
0,1,True,False,False,False,False,False,False,False,False,False,False,False
1,2,True,False,False,False,False,False,False,False,False,False,False,False
2,3,False,False,True,False,False,False,False,False,False,False,False,False
3,4,False,True,False,False,False,False,False,False,False,False,False,False
4,5,True,False,False,False,False,False,False,False,False,False,False,False


In [282]:
discount_category_matrix.columns

Index(['discount_id', 'category_1', 'category_2', 'category_3', 'category_4',
       'category_5', 'category_6', 'category_7', 'category_8', 'category_9',
       'category_10', 'category_11', 'category_12'],
      dtype='object')

In [283]:
discount_category_matrix.shape

(1024, 13)

### Step 5: Create Location Features

In [284]:
location_discount_df.head()

,id,location_id,discount_id
0,1,14,1
1,2,17,2
2,3,14,2
3,1301,58,660
4,1316,25,669


In [285]:
discount_locations = location_discount_df[[
    'location_id',
    'discount_id'
]].copy()

In [286]:
discount_locations.head()

,location_id,discount_id
0,14,1
1,17,2
2,14,2
3,58,660
4,25,669


In [287]:
discount_locations.isna().sum()

location_id    0
discount_id    0
dtype: int64

In [288]:
location_features = pd.get_dummies(
    discount_locations['location_id'],
    prefix='location'
)

In [289]:
location_features.head()

,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,location_49,location_50,location_52,location_53,location_54,location_55,location_56,location_58,location_59,location_60,location_62,location_63,location_64,location_67,location_68,location_69,location_70,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [290]:
discount_location_matrix = pd.concat(
    [
        discount_locations[['discount_id']],
        location_features
    ],
    axis=1
)

In [291]:
discount_location_matrix.head()

,discount_id,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,location_49,location_50,location_52,location_53,location_54,location_55,location_56,location_58,location_59,location_60,location_62,location_63,location_64,location_67,location_68,location_69,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,1,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,660,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,669,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [292]:
discount_location_matrix = (discount_location_matrix.groupby('discount_id').max().reset_index())

In [293]:
discount_location_matrix.head()

,discount_id,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,location_49,location_50,location_52,location_53,location_54,location_55,location_56,location_58,location_59,location_60,location_62,location_63,location_64,location_67,location_68,location_69,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,1,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,False,False,False,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,3,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,4,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,5,False,False,False,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [294]:
discount_location_matrix.shape

(1024, 220)

### Step 6: Merge Item Features into item_features_raw

In [295]:
discount_base.head()

,discount_id,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived
0,785,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False
1,613,нутріциолог,PROMOCODE,15.0,15.0,2,0,False
2,765,English School,PROMOCODE,20.0,20.0,1,0,False
3,47,Sightseeing,PROMOCODE,10.0,10.0,0,0,False
4,215,Cafe,PROMOCODE,5.0,5.0,0,0,False


In [296]:
discount_category_matrix.shape

(1024, 13)

In [297]:
discount_base.columns

Index(['discount_id', 'type', 'discount_type', 'size_min', 'size_max',
       'like_count', 'dislike_count', 'is_archived'],
      dtype='object')

In [298]:
discount_category_matrix.columns

Index(['discount_id', 'category_1', 'category_2', 'category_3', 'category_4',
       'category_5', 'category_6', 'category_7', 'category_8', 'category_9',
       'category_10', 'category_11', 'category_12'],
      dtype='object')

In [299]:
discount_location_matrix.columns

Index(['discount_id', 'location_10', 'location_11', 'location_12',
       'location_13', 'location_14', 'location_15', 'location_16',
       'location_17', 'location_23',
       ...
       'location_241', 'location_242', 'location_243', 'location_244',
       'location_245', 'location_246', 'location_247', 'location_248',
       'location_249', 'location_250'],
      dtype='object', length=220)

In [300]:
item_features_raw = discount_base.merge(discount_category_matrix,on='discount_id',how='left')

In [301]:
item_features_raw.head()

,discount_id,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12
0,785,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,True
1,613,нутріциолог,PROMOCODE,15.0,15.0,2,0,False,True,False,False,False,True,False,False,False,False,False,False,True
2,765,English School,PROMOCODE,20.0,20.0,1,0,False,False,False,False,False,True,False,False,False,False,False,False,False
3,47,Sightseeing,PROMOCODE,10.0,10.0,0,0,False,False,False,False,False,False,False,False,True,False,False,False,False
4,215,Cafe,PROMOCODE,5.0,5.0,0,0,False,True,False,False,False,False,False,False,True,False,False,False,False


In [302]:
item_features_raw = item_features_raw.merge(discount_location_matrix,on='discount_id',how='left')

In [303]:
item_features_raw.head()

,discount_id,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,785,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,613,нутріциолог,PROMOCODE,15.0,15.0,2,0,False,True,False,False,False,True,False,False,False,False,False,False,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,765,English School,PROMOCODE,20.0,20.0,1,0,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,47,Sightseeing,PROMOCODE,10.0,10.0,0,0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,215,Cafe,PROMOCODE,5.0,5.0,0,0,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [304]:
item_features_raw.isna().sum().sort_values(ascending=False).head()

discount_id     0
location_175    0
location_163    0
location_164    0
location_165    0
dtype: int64

## 7. Build Final Item-Feature Matrix

In [305]:
item_features = item_features_raw.copy()

In [306]:
item_features.head()

,discount_id,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,785,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,613,нутріциолог,PROMOCODE,15.0,15.0,2,0,False,True,False,False,False,True,False,False,False,False,False,False,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,765,English School,PROMOCODE,20.0,20.0,1,0,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,47,Sightseeing,PROMOCODE,10.0,10.0,0,0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,215,Cafe,PROMOCODE,5.0,5.0,0,0,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [307]:
item_features.dtypes

discount_id        int64
type              object
discount_type     object
size_min         float64
size_max         float64
                  ...   
location_246        bool
location_247        bool
location_248        bool
location_249        bool
location_250        bool
Length: 239, dtype: object

In [308]:
discount_ids = item_features['discount_id']

In [309]:
item_features = item_features.drop(columns=['discount_id'])

In [310]:
item_features.head()

,type,discount_type,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,...,location_211,location_212,location_213,location_214,location_215,location_216,location_217,location_218,location_219,location_220,location_221,location_222,location_223,location_224,location_225,location_226,location_227,location_228,location_229,location_230,location_231,location_232,location_233,location_234,location_235,location_236,location_237,location_238,location_239,location_240,location_241,location_242,location_243,location_244,location_245,location_246,location_247,location_248,location_249,location_250
0,shoe and accessory cleaning,PROMOCODE,10.0,10.0,1,0,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,нутріциолог,PROMOCODE,15.0,15.0,2,0,False,True,False,False,False,True,False,False,False,False,False,False,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,English School,PROMOCODE,20.0,20.0,1,0,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,Sightseeing,PROMOCODE,10.0,10.0,0,0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,Cafe,PROMOCODE,5.0,5.0,0,0,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [311]:
item_features = pd.get_dummies(
    item_features,
    columns=['type','discount_type'],
    drop_first=False
)

In [312]:
bool_cols = item_features.select_dtypes(include=['bool']).columns
item_features[bool_cols] = item_features[bool_cols].astype(int)

In [313]:
item_features.head()

,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
0,10.0,10.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,15.0,15.0,2,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,20.0,20.0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,10.0,10.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,5.0,5.0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [314]:
item_features.dtypes.value_counts()

int32      885
float64      2
int64        2
Name: count, dtype: int64

##### Scale numeric columns

In [315]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler

In [316]:
numeric_cols = [
    'size_min',
    'size_max',
    'like_count',
    'dislike_count'
]

scaler = StandardScaler()

item_features[numeric_cols] = scaler.fit_transform(
    item_features[numeric_cols]
) 

In [317]:
item_features.head()

,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
0,-0.125576,-0.281628,-0.137640,-0.145236,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0.543524,0.276835,0.110604,-0.145236,0,1,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1.212624,0.835299,-0.137640,-0.145236,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,-0.125576,-0.281628,-0.385883,-0.145236,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,-0.794676,-0.840091,-0.385883,-0.145236,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [318]:
item_features_matrix = item_features.copy()
item_features_matrix.index = discount_ids

In [319]:
item_features.head()

,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
0,-0.125576,-0.281628,-0.137640,-0.145236,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0.543524,0.276835,0.110604,-0.145236,0,1,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1.212624,0.835299,-0.137640,-0.145236,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,-0.125576,-0.281628,-0.385883,-0.145236,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,-0.794676,-0.840091,-0.385883,-0.145236,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [320]:
item_features_matrix.index.name = 'discount_id'

In [321]:
item_features_matrix.columns

Index(['size_min', 'size_max', 'like_count', 'dislike_count', 'is_archived',
       'category_1', 'category_2', 'category_3', 'category_4', 'category_5',
       ...
       'type_услуги фотографа', 'type_художественная мастерская',
       'type_цветочный', 'type_шоколад ручной работы',
       'type_языковая онлайн школа', 'type_інтернет-магазин',
       'type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞', 'discount_type_OTHER',
       'discount_type_PROMOCODE', 'discount_type_SHOW_AT_LOCATION'],
      dtype='object', length=889)

In [322]:
item_features.head(5)

,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
0,-0.125576,-0.281628,-0.137640,-0.145236,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0.543524,0.276835,0.110604,-0.145236,0,1,0,0,0,1,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,1.212624,0.835299,-0.137640,-0.145236,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,-0.125576,-0.281628,-0.385883,-0.145236,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,-0.794676,-0.840091,-0.385883,-0.145236,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [323]:
item_features.shape

(909, 889)

In [324]:
item_features_matrix.columns

Index(['size_min', 'size_max', 'like_count', 'dislike_count', 'is_archived',
       'category_1', 'category_2', 'category_3', 'category_4', 'category_5',
       ...
       'type_услуги фотографа', 'type_художественная мастерская',
       'type_цветочный', 'type_шоколад ручной работы',
       'type_языковая онлайн школа', 'type_інтернет-магазин',
       'type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞', 'discount_type_OTHER',
       'discount_type_PROMOCODE', 'discount_type_SHOW_AT_LOCATION'],
      dtype='object', length=889)

In [325]:
item_features_matrix.isna().sum().sum()

0

In [326]:
item_features_matrix.dtypes.value_counts()

int32      885
float64      4
Name: count, dtype: int64

In [327]:
item_features_matrix.select_dtypes(include=["object"]).columns

Index([], dtype='object')

In [328]:
item_features_matrix.shape

(909, 889)

#### 8 Build User-Item Interaction Weights


In [329]:
interaction_clean.head()

,user_id,discount_id,event_type,event_time,interaction_weight,source_table,location_id,role_id
0,1535,17,SHOW_AT_LOCATION,2025-01-16 15:18:53.386,0.1,discount_button_click_statistics,16.0,3.0
1,9,522,SHOW_AT_LOCATION,2025-01-16 15:39:03.441,0.1,discount_button_click_statistics,16.0,3.0
2,1455,5,SHOW_AT_LOCATION,2025-01-16 17:59:54.167,0.1,discount_button_click_statistics,16.0,3.0
3,1455,2,SHOW_AT_LOCATION,2025-01-16 18:00:05.394,0.1,discount_button_click_statistics,16.0,3.0
4,1418,11,SHOW_AT_LOCATION,2025-01-16 19:02:26.925,0.1,discount_button_click_statistics,16.0,3.0


In [330]:
interaction_clean.columns

Index(['user_id', 'discount_id', 'event_type', 'event_time',
       'interaction_weight', 'source_table', 'location_id', 'role_id'],
      dtype='object')

In [331]:
valid_item_ids = set(item_features_matrix.index)


In [332]:
len(valid_item_ids)

909

In [333]:
interaction_for_profiles = interaction_clean[
    interaction_clean["discount_id"].isin(valid_item_ids)
].copy()


In [334]:
interaction_for_profiles.head()

,user_id,discount_id,event_type,event_time,interaction_weight,source_table,location_id,role_id
0,1535,17,SHOW_AT_LOCATION,2025-01-16 15:18:53.386,0.1,discount_button_click_statistics,16.0,3.0
1,9,522,SHOW_AT_LOCATION,2025-01-16 15:39:03.441,0.1,discount_button_click_statistics,16.0,3.0
2,1455,5,SHOW_AT_LOCATION,2025-01-16 17:59:54.167,0.1,discount_button_click_statistics,16.0,3.0
3,1455,2,SHOW_AT_LOCATION,2025-01-16 18:00:05.394,0.1,discount_button_click_statistics,16.0,3.0
4,1418,11,SHOW_AT_LOCATION,2025-01-16 19:02:26.925,0.1,discount_button_click_statistics,16.0,3.0


In [335]:
interaction_for_profiles.shape

(32164, 8)

In [336]:
user_item_weights = (
    interaction_for_profiles.groupby(['user_id','discount_id'])['interaction_weight'].sum().reset_index()
)

In [337]:
user_item_weights.head()

,user_id,discount_id,interaction_weight
0,1,684,2.0
1,3,2,3.0
2,3,5,3.0
3,3,17,3.0
4,3,21,3.0


In [338]:
user_item_weights = user_item_weights.rename(columns={
    'interaction_weight':'interaction_score'})



In [339]:
user_item_weights.head()

,user_id,discount_id,interaction_score
0,1,684,2.0
1,3,2,3.0
2,3,5,3.0
3,3,17,3.0
4,3,21,3.0


In [340]:
user_item_weights.shape

(18095, 3)

In [341]:
user_item_weights['interaction_score'].describe()

count    18095.000000
mean         1.917994
std          2.761904
min         -3.000000
25%          0.300000
50%          0.600000
75%          2.600000
max         70.300000
Name: interaction_score, dtype: float64

In [342]:
user_item_weights.isna().sum()

user_id              0
discount_id          0
interaction_score    0
dtype: int64

In [343]:
user_item_weights.head()

,user_id,discount_id,interaction_score
0,1,684,2.0
1,3,2,3.0
2,3,5,3.0
3,3,17,3.0
4,3,21,3.0


In [344]:
user_item_weights.shape

(18095, 3)

In [345]:
user_item_weights['interaction_score'].describe()

count    18095.000000
mean         1.917994
std          2.761904
min         -3.000000
25%          0.300000
50%          0.600000
75%          2.600000
max         70.300000
Name: interaction_score, dtype: float64

### 9. Build User Profile Matrix


In [346]:
item_features_matrix = item_features_matrix.copy()

In [347]:
item_features_matrix.shape

(909, 889)

In [348]:
valid_discount_ids = set(item_features_matrix.index)

user_item_weights_filtered = user_item_weights[
    user_item_weights["discount_id"].isin(valid_discount_ids)
].copy()

In [349]:
user_item_weights_filtered.shape

(18095, 3)

In [350]:
user_item_features = user_item_weights_filtered.merge(
    item_features_matrix,
    left_on="discount_id",
    right_index=True,
    how="left"
)

In [351]:
user_item_features.head()

,user_id,discount_id,interaction_score,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
0,1,684,2.0,1.212624,0.835299,-0.137640,-0.145236,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,3,2,3.0,1.212624,0.835299,10.785071,-0.145236,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,3,5,3.0,1.212624,0.835299,13.515749,10.010080,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,3,17,3.0,1.881724,1.393762,11.778045,4.932422,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,3,21,3.0,-0.125576,-0.281628,1.103577,-0.145236,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [352]:
user_item_features.shape

(18095, 892)

In [353]:
feature_cols = user_item_features.columns.difference([
    "user_id",
    "discount_id",
    "interaction_score"
])

In [354]:
len(feature_cols)

889

In [355]:
weighted_item_features = user_item_features[feature_cols].multiply(
    user_item_features["interaction_score"],
    axis=0
)

In [356]:
weighted_item_features["user_id"] = user_item_features["user_id"]

C:\Users\Seidahmet\AppData\Local\Temp\ipykernel_34120\2102308407.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weighted_item_features["user_id"] = user_item_features["user_id"]


In [357]:
user_profile_matrix = (
    weighted_item_features
    .groupby("user_id")
    .sum()
)

In [358]:
# Reorder user profile columns to match item feature matrix

user_profile_matrix = user_profile_matrix[item_features_matrix.columns]

In [359]:
user_strength = (
    user_item_weights_filtered
    .groupby("user_id")["interaction_score"]
    .apply(lambda x: x.abs().sum())
)

In [360]:
user_profile_matrix = user_profile_matrix.div(
    user_strength,
    axis=0
)

In [361]:
user_profile_matrix.shape

(2458, 889)

In [362]:
user_profile_matrix.head()

,size_min,size_max,like_count,dislike_count,is_archived,category_1,category_2,category_3,category_4,category_5,category_6,category_7,category_8,category_9,category_10,category_11,category_12,location_10,location_11,location_12,location_13,location_14,location_15,location_16,location_17,location_23,location_24,location_25,location_26,location_27,location_28,location_29,location_30,location_31,location_43,location_44,location_45,location_46,location_47,location_48,...,type_картинг,type_кафе,"type_кафе, пекарня",type_компьютерный клуб,type_кондитер,type_косметика,type_магазин брэндовой одежды,type_магазин впечатлений,type_магазин жіночого одягу,type_магазин китайского чая,"type_маникюр, педикюр, подология",type_маркетплэйс цветов и подарков,"type_массаж, остеопатия",type_мастер-классы,type_нутріциолог,type_одяг для дітей,type_онлайн магазин,type_пекарня,type_психология,type_сервис доставки еды,type_сеть городских кафе,type_сладости ручной работы,"type_сладости, ПП десерты","type_спа, массаж, косметология",type_студия аэродизайна,"type_суши, роллы, доставка","type_суши,пицца,выпечка",type_таро,type_тату,type_услуги массажа,type_услуги фотографа,type_художественная мастерская,type_цветочный,type_шоколад ручной работы,type_языковая онлайн школа,type_інтернет-магазин,type_𝐃𝐨𝐦𝐨𝐰𝐞 𝐣𝐞𝐝𝐳𝐞𝐧𝐢𝐞,discount_type_OTHER,discount_type_PROMOCODE,discount_type_SHOW_AT_LOCATION
user_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1.212624,0.835299,-0.137640,-0.145236,0.0,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.00000,0.00000,1.00000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0
3,0.541618,0.373890,3.710487,1.156728,0.0,0.857550,0.000000,0.000000,0.000000,0.19943,0.000000,0.000000,0.000000,0.142450,0.000000,0.000000,0.199430,0.19943,0.19943,0.19943,0.256410,0.256410,0.256410,0.256410,0.683761,0.0,0.0,0.0,0.0,0.0,0.116809,0.0,0.0,0.0,0.19943,0.19943,0.19943,0.19943,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.05698,0.0,0.0,0.000000,0.000000,0.08547,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0
4,1.435657,1.021453,12.026288,4.932422,0.0,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0
7,0.085009,-0.105863,3.366634,0.891289,0.0,0.390181,0.112403,0.078811,0.000000,0.00000,0.038760,0.000000,0.111111,0.160207,0.025840,0.000000,0.169251,0.00000,0.00000,0.00000,0.666667,0.998708,0.666667,0.666667,0.667959,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.001292,0.002584,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.001292,0.0,0.00000,0.0,0.0,0.001292,0.077519,0.00000,0.206718,0.0,0.0,0.0,0.0,0.0,0.000000,0.03876,0.0,0.0,0.025840,0.0,0.000000,0.025840,0.0,0.0,0.0,0.0,1.0,0.0
8,0.374141,0.234913,3.595669,1.337719,0.0,0.426849,0.071781,0.078356,0.054795,0.00000,0.038904,0.001096,0.103562,0.223014,0.032877,0.012055,0.046027,0.00000,0.00000,0.00000,0.578082,0.578082,0.989041,0.589041,0.590685,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.00000,0.00000,0.00000,0.00000,0.0,0.0,...,0.0,0.000000,0.000000,0.0,0.0,0.010959,0.0,0.0,0.0,0.0,0.0,0.016438,0.000000,0.0,0.0000

In [363]:
user_profile_matrix.isna().sum().sum()

0

In [364]:
user_profile_matrix.columns.equals(item_features_matrix.columns)

True

### Step 10: Compute Cosine Similarity

In [365]:
from sklearn.metrics.pairwise import cosine_similarity

In [366]:
similarity_matrix = cosine_similarity(
    user_profile_matrix,item_features_matrix   
)

In [367]:
similarity_matrix.shape

(2458, 909)

In [368]:
similarity_df = pd.DataFrame(similarity_matrix,
                             index = user_profile_matrix.index,
                             columns=item_features_matrix.index)

In [370]:
similarity_df.head()

discount_id,785,613,765,47,215,247,253,526,541,236,251,199,200,219,1216,487,530,520,486,492,509,1285,910,684,864,751,1286,869,874,873,220,506,572,782,831,944,555,1217,1287,183,...,1108,1278,1198,1279,1109,1110,1111,1199,1112,1200,1113,1202,1201,1281,367,1114,1203,1282,1115,1204,1283,1116,1205,766,1117,1118,1207,1208,1119,1209,1210,1211,440,1123,925,1213,1124,1125,26,335
user_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0.203646,0.396358,0.431283,0.123846,-0.086317,-0.093826,0.123846,-0.080362,-0.093826,0.388252,0.080412,0.518104,-0.093826,0.105608,0.111466,0.479586,0.607513,0.395426,0.251393,0.302683,-0.082772,0.111466,0.250990,1.000000,-0.242093,0.123846,0.123846,0.507332,0.150808,0.080412,0.119518,0.304198,0.445817,0.493196,0.094891,-0.062771,0.234360,0.123846,0.127986,0.057642,...,0.336495,0.123846,0.250990,0.123846,0.087264,0.102184,0.102184,-0.191952,0.098122,0.424369,-0.046817,0.123846,-0.228661,0.479699,0.087676,0.046978,0.123846,0.152605,0.304198,-0.100442,0.119429,0.479699,0.665228,-0.025676,-0.282897,0.304198,-0.093826,0.304198,0.198929,0.343336,0.493787,0.111466,-0.018947,0.311787,-0.371623,0.273280,0.136546,0.095009,0.016608,0.310821
3,0.035539,0.287493,0.196430,-0.086622,-0.044154,-0.132549,-0.070661,-0.101498,-0.132549,0.040611,0.031177,-0.009241,-0.121032,0.332937,-0.057852,0.119832,0.264321,0.329873,0.407164,0.039531,0.173576,0.008513,-0.071470,0.145585,-0.154139,-0.070661,-0.010012,0.139584,0.006674,0.035539,0.030130,-0.073534,0.070723,0.073403,-0.054141,0.021614,0.078093,-0.010012,0.128060,-0.069926,...,0.271912,-0.070661,-0.071470,-0.010012,0.564733,-0.053034,-0.053034,0.141037,0.032454,0.427786,0.255702,0.009459,-0.080410,0.136323,0.511580,-0.073388,-0.169934,-0.033848,-0.086622,0.068924,0.573009,0.033940,0.091896,0.280319,-0.030253,-0.086622,-0.118504,-0.086622,0.321155,-0.021923,0.120094,-0.066184,0.003323,0.113385,-0.149554,0.028857,0.141479,0.287720,0.869049,0.089643
4,-0.045163,0.074517,0.024959,-0.173363,-0.155800,-0.200974,-0.173363,-0.172134,-0.200974,-0.046327,-0.045163,-0.114688,-0.200974,0.184802,-0.156033,0.036406,0.214072,0.258186,0.351114,-0.067127,0.126548,-0.123695,-0.143040,0.033786,-0.203527,-0.173363,-0.137433,0.080526,-0.050376,-0.045163,-0.067127,-0.173363,-0.048291,-0.005041,-0.132831,0.004778,-0.007463,-0.137433,0.060702,-0.175211,...,0.129988,-0.173363,-0.143040,-0.137433,0.407859,-0.143040,-0.143040,0.100346,-0.055110,0.454225,0.168378,-0.137433,-0.167861,0.006986,0.413099,-0.145500,-0.173363,-0.103827,-0.173363,-0.043860,0.489260,-0.051962,0.025326,0.229002,-0.037071,-0.173363,-0.200974,-0.173363,0.135761,-0.114688,0.003819,-0.156033,-0.066541,-0.069018,-0.203527,-0.004165,-0.007817,0.166254,0.797050,-0.040787
7,0.048266,0.121708,0.028430,-0.035383,0.009035,-0.030769,-0.029560,-0.012147,-0.030769,0.053740,0.047533,-0.038257,-0.042501,0.295787,-0.031846,0.052519,0.129989,0.245254,0.392461,0.071738,0.222460,-0.002055,-0.032355,0.061116,-0.024920,-0.029560,0.030662,0.023233,-0.012695,0.034549,0.064735,-0.039214,-0.022941,-0.010300,-0.012552,0.045673,0.033858,0.043841,0.300567,-0.040505,...,0.243671,-0.029560,-0.032355,0.043841,0.537049,-0.040067,-0.040067,0.391816,0.053274,0.280608,0.207281,-0.002284,-0.001189,0.094859,0.460682,-0.017980,-0.078749,-0.106285,-0.039214,0.156902,0.481970,-0.040272,-0.021853,0.508631,0.206313,-0.039214,-0.025780,-0.039214,0.272667,-0.034396,0.007073,-0.027571,0.041546,-0.017393,-0.086441,-0.058004,0.100703,0.266100,0.826823,0.057181
8,0.018404,0.121235,0.060339,-0.065361,-0.048860,-0.097719,-0.052159,-0.057906,-0.097719,0.045723,0.031770,-0.025174,-0.104595,0.266963,-0.058827,0.094586,0.211775,0.283985,0.385133,0.027354,0.176519,-0.026668,-0.056227,0.092253,-0.110036,-0.052159,-0.005466,0.094814,-0.004252,0.014928,0.033812,-0.068147,0.024531,0.049777,-0.026554,0.031577,0.043093,-0.000076,0.267184,-0.061084,...,0.248547,-0.052159,-0.056227,-0.000076,0.496120,-0.063372,-0.063372,0.302832,0.029880,0.404170,0.196241,-0

In [371]:

similarity_df.head()

discount_id,785,613,765,47,215,247,253,526,541,236,251,199,200,219,1216,487,530,520,486,492,509,1285,910,684,864,751,1286,869,874,873,220,506,572,782,831,944,555,1217,1287,183,...,1108,1278,1198,1279,1109,1110,1111,1199,1112,1200,1113,1202,1201,1281,367,1114,1203,1282,1115,1204,1283,1116,1205,766,1117,1118,1207,1208,1119,1209,1210,1211,440,1123,925,1213,1124,1125,26,335
user_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0.203646,0.396358,0.431283,0.123846,-0.086317,-0.093826,0.123846,-0.080362,-0.093826,0.388252,0.080412,0.518104,-0.093826,0.105608,0.111466,0.479586,0.607513,0.395426,0.251393,0.302683,-0.082772,0.111466,0.250990,1.000000,-0.242093,0.123846,0.123846,0.507332,0.150808,0.080412,0.119518,0.304198,0.445817,0.493196,0.094891,-0.062771,0.234360,0.123846,0.127986,0.057642,...,0.336495,0.123846,0.250990,0.123846,0.087264,0.102184,0.102184,-0.191952,0.098122,0.424369,-0.046817,0.123846,-0.228661,0.479699,0.087676,0.046978,0.123846,0.152605,0.304198,-0.100442,0.119429,0.479699,0.665228,-0.025676,-0.282897,0.304198,-0.093826,0.304198,0.198929,0.343336,0.493787,0.111466,-0.018947,0.311787,-0.371623,0.273280,0.136546,0.095009,0.016608,0.310821
3,0.035539,0.287493,0.196430,-0.086622,-0.044154,-0.132549,-0.070661,-0.101498,-0.132549,0.040611,0.031177,-0.009241,-0.121032,0.332937,-0.057852,0.119832,0.264321,0.329873,0.407164,0.039531,0.173576,0.008513,-0.071470,0.145585,-0.154139,-0.070661,-0.010012,0.139584,0.006674,0.035539,0.030130,-0.073534,0.070723,0.073403,-0.054141,0.021614,0.078093,-0.010012,0.128060,-0.069926,...,0.271912,-0.070661,-0.071470,-0.010012,0.564733,-0.053034,-0.053034,0.141037,0.032454,0.427786,0.255702,0.009459,-0.080410,0.136323,0.511580,-0.073388,-0.169934,-0.033848,-0.086622,0.068924,0.573009,0.033940,0.091896,0.280319,-0.030253,-0.086622,-0.118504,-0.086622,0.321155,-0.021923,0.120094,-0.066184,0.003323,0.113385,-0.149554,0.028857,0.141479,0.287720,0.869049,0.089643
4,-0.045163,0.074517,0.024959,-0.173363,-0.155800,-0.200974,-0.173363,-0.172134,-0.200974,-0.046327,-0.045163,-0.114688,-0.200974,0.184802,-0.156033,0.036406,0.214072,0.258186,0.351114,-0.067127,0.126548,-0.123695,-0.143040,0.033786,-0.203527,-0.173363,-0.137433,0.080526,-0.050376,-0.045163,-0.067127,-0.173363,-0.048291,-0.005041,-0.132831,0.004778,-0.007463,-0.137433,0.060702,-0.175211,...,0.129988,-0.173363,-0.143040,-0.137433,0.407859,-0.143040,-0.143040,0.100346,-0.055110,0.454225,0.168378,-0.137433,-0.167861,0.006986,0.413099,-0.145500,-0.173363,-0.103827,-0.173363,-0.043860,0.489260,-0.051962,0.025326,0.229002,-0.037071,-0.173363,-0.200974,-0.173363,0.135761,-0.114688,0.003819,-0.156033,-0.066541,-0.069018,-0.203527,-0.004165,-0.007817,0.166254,0.797050,-0.040787
7,0.048266,0.121708,0.028430,-0.035383,0.009035,-0.030769,-0.029560,-0.012147,-0.030769,0.053740,0.047533,-0.038257,-0.042501,0.295787,-0.031846,0.052519,0.129989,0.245254,0.392461,0.071738,0.222460,-0.002055,-0.032355,0.061116,-0.024920,-0.029560,0.030662,0.023233,-0.012695,0.034549,0.064735,-0.039214,-0.022941,-0.010300,-0.012552,0.045673,0.033858,0.043841,0.300567,-0.040505,...,0.243671,-0.029560,-0.032355,0.043841,0.537049,-0.040067,-0.040067,0.391816,0.053274,0.280608,0.207281,-0.002284,-0.001189,0.094859,0.460682,-0.017980,-0.078749,-0.106285,-0.039214,0.156902,0.481970,-0.040272,-0.021853,0.508631,0.206313,-0.039214,-0.025780,-0.039214,0.272667,-0.034396,0.007073,-0.027571,0.041546,-0.017393,-0.086441,-0.058004,0.100703,0.266100,0.826823,0.057181
8,0.018404,0.121235,0.060339,-0.065361,-0.048860,-0.097719,-0.052159,-0.057906,-0.097719,0.045723,0.031770,-0.025174,-0.104595,0.266963,-0.058827,0.094586,0.211775,0.283985,0.385133,0.027354,0.176519,-0.026668,-0.056227,0.092253,-0.110036,-0.052159,-0.005466,0.094814,-0.004252,0.014928,0.033812,-0.068147,0.024531,0.049777,-0.026554,0.031577,0.043093,-0.000076,0.267184,-0.061084,...,0.248547,-0.052159,-0.056227,-0.000076,0.496120,-0.063372,-0.063372,0.302832,0.029880,0.404170,0.196241,-0

In [372]:
similarity_df.shape

(2458, 909)

In [373]:
similarity_df.isna().sum().sum()

0

In [374]:
###Check one user's score
sample_user_id = similarity_df.index[0]
similarity_df.loc[sample_user_id].sort_values(ascending=False).head(10)

discount_id
684     1.000000
625     0.707462
1205    0.665228
626     0.650167
627     0.646959
291     0.646233
714     0.637226
956     0.629073
940     0.629073
734     0.627645
Name: 1, dtype: float64

### 11: Remove Already Interacted Discounts

In [375]:
# Convert similarity matrix to long format

recommendation_scores = (
    similarity_df
    .reset_index()
    .melt(
        id_vars="user_id",
        var_name="discount_id",
        value_name="similarity_score"
    )
)

In [376]:
recommendation_scores.head()

,user_id,discount_id,similarity_score
0,1,785,0.203646
1,3,785,0.035539
2,4,785,-0.045163
3,7,785,0.048266
4,8,785,0.018404


In [377]:
recommendation_scores.shape

(2234322, 3)

In [380]:
interacted_pairs = user_item_weights[['user_id','discount_id']].drop_duplicates()

In [381]:
interacted_pairs.head()

,user_id,discount_id
0,1,684
1,3,2
2,3,5
3,3,17
4,3,21


In [382]:
interacted_pairs.head()

,user_id,discount_id
0,1,684
1,3,2
2,3,5
3,3,17
4,3,21


In [383]:
recommendation_scores = recommendation_scores.merge(
    interacted_pairs,
    on=['user_id','discount_id'],
    how='left',
    indicator=True
)

In [386]:
recommendation_scores["_merge"].value_counts()

_merge
left_only     2216227
both            18095
right_only          0
Name: count, dtype: int64

In [387]:
# Keep only discounts the user has not interacted with

recommendation_candidates = recommendation_scores[
    recommendation_scores["_merge"] == "left_only"
].copy()

In [388]:
recommendation_candidates = recommendation_candidates.drop(columns=['_merge'])

In [389]:
recommendation_candidates.shape

(2216227, 3)

In [390]:
recommendation_candidates.head(3)

,user_id,discount_id,similarity_score
0,1,785,0.203646
1,3,785,0.035539
2,4,785,-0.045163


In [391]:
recommendation_candidates.isna().sum()

user_id             0
discount_id         0
similarity_score    0
dtype: int64

In [392]:
recommendation_candidates[
    recommendation_candidates["user_id"] == recommendation_candidates["user_id"].iloc[0]
].sort_values("similarity_score", ascending=False).head(10)

,user_id,discount_id,similarity_score
204014,1,625,0.707462
2190078,1,1205,0.665228
835720,1,626,0.650167
319540,1,627,0.646959
943872,1,291,0.646233
329372,1,714,0.637226
304792,1,940,0.629073
1496922,1,956,0.629073
1415808,1,775,0.627645
457188,1,528,0.627645


### 12. Generate Top-N Recommendations

In [394]:
N = 15

In [395]:
# Sort recommendations by user and similarity score

recommendation_candidates_sorted = recommendation_candidates.sort_values(
    by=["user_id", "similarity_score"],
    ascending=[True, False]
)

In [396]:
# Create recommendation rank for each user

recommendation_candidates_sorted["rank"] = (
    recommendation_candidates_sorted
    .groupby("user_id")
    .cumcount() + 1
)

In [397]:
# Keep only top-N recommendations

top_n_recommendations = recommendation_candidates_sorted[
    recommendation_candidates_sorted["rank"] <= N
].copy()

In [399]:
top_n_recommendations


,user_id,discount_id,similarity_score,rank
204014,1,625,0.707462,1
2190078,1,1205,0.665228,2
835720,1,626,0.650167,3
319540,1,627,0.646959,4
943872,1,291,0.646233,5
...,...,...,...,...
324455,5576,600,0.676115,11
1179839,5576,36,0.669042,12
1509211,5576,14,0.669029,13
1300281,5576,569,0.636062,14


In [400]:
top_n_recommendations.head(20)

,user_id,discount_id,similarity_score,rank
204014,1,625,0.707462,1
2190078,1,1205,0.665228,2
835720,1,626,0.650167,3
319540,1,627,0.646959,4
943872,1,291,0.646233,5
329372,1,714,0.637226,6
304792,1,940,0.629073,7
1496922,1,956,0.629073,8
231052,1,534,0.627645,9
457188,1,528,0.627645,10


In [401]:
top_n_recommendations.shape

(36870, 4)

In [402]:
top_n_recommendations['user_id'].nunique()

2458

In [403]:
top_n_recommendations.groupby('user_id').size().describe()

count    2458.0
mean       15.0
std         0.0
min        15.0
25%        15.0
50%        15.0
75%        15.0
max        15.0
dtype: float64

In [404]:
sample_user_id = top_n_recommendations['user_id'].iloc[0]

top_n_recommendations[top_n_recommendations['user_id'] == sample_user_id]

,user_id,discount_id,similarity_score,rank
204014,1,625,0.707462,1
2190078,1,1205,0.665228,2
835720,1,626,0.650167,3
319540,1,627,0.646959,4
943872,1,291,0.646233,5
329372,1,714,0.637226,6
304792,1,940,0.629073,7
1496922,1,956,0.629073,8
231052,1,534,0.627645,9
457188,1,528,0.627645,10


In [405]:
top_n_recommendations.groupby('user_id').size().describe()

count    2458.0
mean       15.0
std         0.0
min        15.0
25%        15.0
50%        15.0
75%        15.0
max        15.0
dtype: float64